# Интерактивный Event Study — анализ дивидендных событий\n\nDash-приложение для анализа влияния одного дивидендного события на котировки акции.\n\n**Параметры:**\n- Тикер и конкретное событие (дата объявления дивидендов)\n- Модель ожидаемой доходности: mean adjusted, market model, CAPM\n- Асимметричное событийное окно (дней до / после t=0)\n- Оценочное окно (длина + отступ от событийного окна)\n\n**Визуализация:** кривая CAR + объём торгов + метрики-карточки

In [1]:
import socket
import uuid
from datetime import date

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, html, dcc, Input, Output, State, no_update

from core.nirs import (
    load_events, load_rf, load_prices,
    EventStudyRunner, EventStudyConfig, Event,
)

# === Пути к данным ===
STOCKS_DIR = '../data/stocks'
EVENTS_PATH = '../data/stocks/dividends_all.csv'
RF_PATH = '../data/stocks/RUONIA_RC_F11_01_2010_T13_03_2026.xlsx'

# Загружаем всё один раз (широкий диапазон для любого оценочного окна)
START = '2010-01-01'
END = '2026-12-31'

events_list = load_events(EVENTS_PATH)
all_tickers = sorted({ev.ticker for ev in events_list})

prices = load_prices(STOCKS_DIR, tickers=all_tickers + ['IMOEX'], start_date=START, end_date=END)
market = prices.pop('IMOEX', None)
if market is None:
    imoex_prices = load_prices(STOCKS_DIR, tickers=['IMOEX'], start_date=START, end_date=END)
    market = imoex_prices.get('IMOEX')

rf = load_rf(RF_PATH, start=date(2010, 1, 1), end=date(2026, 12, 31))

runner = EventStudyRunner(prices=prices, market=market, rf=rf)

# Группируем события по тикеру для дропдаунов
events_by_ticker = {}
for ev in events_list:
    events_by_ticker.setdefault(ev.ticker, []).append(ev)

print(f'Тикеров: {len(all_tickers)}')
print(f'Событий: {len(events_list)}')
print(f'IMOEX строк: {len(market) if market is not None else "НЕТ"}')
print(f'RUONIA строк: {len(rf)}')

C:\Users\Ruslan\anaconda3\envs\data-core\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Тикеров: 11
Событий: 110
IMOEX строк: 3795
RUONIA строк: 3980


In [2]:
# === Загрузка объёмов торгов (для barchart) ===
from core.stock_data_provider import get_stock_data

volumes = {}
for ticker in all_tickers:
    try:
        df = get_stock_data(ticker, normalized=True)
        vol = df.set_index('DATE')['VOL']
        vol.index = pd.to_datetime(vol.index)
        volumes[ticker] = vol
    except ValueError:
        pass

print(f'Объёмы загружены для {len(volumes)} тикеров')

Объёмы загружены для 11 тикеров


In [3]:
# === Dash-приложение ===

def _find_free_port(min_port=10001):
    port = min_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('localhost', port)) != 0:
                return port
        port += 1


def _event_label(ev: Event) -> str:
    return f"Анонс дивидендов ({ev.event_date.strftime('%d.%m.%Y')}, {ev.dividend:.0f} руб/акция)"


def _event_value(ev: Event) -> str:
    return f"{ev.ticker}_{ev.event_date.isoformat()}"


# Начальный тикер
default_ticker = all_tickers[0]
default_events = events_by_ticker.get(default_ticker, [])

app = Dash(f'event_study_{uuid.uuid4().hex[:8]}')

app.layout = html.Div([
    html.H2('Event Study — анализ дивидендных событий', style={'marginBottom': '20px'}),

    # --- Панель управления ---
    html.Div([
        # Строка 1: тикер + событие + модель
        html.Div([
            html.Div([
                html.Label('Компания'),
                dcc.Dropdown(
                    id='ticker-dropdown',
                    options=[{'label': t, 'value': t} for t in all_tickers],
                    value=default_ticker,
                    clearable=False,
                    style={'width': '100%'},
                ),
            ], style={'flex': '1', 'marginRight': '10px'}),

            html.Div([
                html.Label('Событие'),
                dcc.Dropdown(
                    id='event-dropdown',
                    options=[{'label': _event_label(ev), 'value': _event_value(ev)} for ev in default_events],
                    value=_event_value(default_events[0]) if default_events else None,
                    clearable=False,
                    style={'width': '100%'},
                ),
            ], style={'flex': '3', 'marginRight': '10px'}),

            html.Div([
                html.Label('Модель'),
                dcc.Dropdown(
                    id='model-dropdown',
                    options=[
                        {'label': 'Mean Adjusted', 'value': 'mean_adjusted'},
                        {'label': 'Market Model', 'value': 'market_model'},
                        {'label': 'CAPM', 'value': 'capm'},
                    ],
                    value='market_model',
                    clearable=False,
                    style={'width': '100%'},
                ),
            ], style={'flex': '1'}),
        ], style={'display': 'flex', 'marginBottom': '15px'}),

        # Строка 2: слайдеры
        html.Div([
            html.Div([
                html.Label('Событийное окно — дней ДО'),
                dcc.Slider(id='ew-before', min=1, max=40, step=1, value=10,
                           marks={i: str(i) for i in [1, 5, 10, 20, 30, 40]},
                           tooltip={'placement': 'bottom', 'always_visible': True}),
            ], style={'flex': '1', 'marginRight': '15px'}),

            html.Div([
                html.Label('Событийное окно — дней ПОСЛЕ'),
                dcc.Slider(id='ew-after', min=1, max=40, step=1, value=10,
                           marks={i: str(i) for i in [1, 5, 10, 20, 30, 40]},
                           tooltip={'placement': 'bottom', 'always_visible': True}),
            ], style={'flex': '1', 'marginRight': '15px'}),

            html.Div([
                html.Label('Оценочное окно — длина'),
                dcc.Slider(id='est-length', min=30, max=500, step=10, value=200,
                           marks={i: str(i) for i in [30, 100, 200, 300, 400, 500]},
                           tooltip={'placement': 'bottom', 'always_visible': True}),
            ], style={'flex': '1'}),
        ], style={'display': 'flex', 'marginBottom': '20px'}),

        # Строка 3: кнопка
        html.Div([
            html.Button('Рассчитать', id='calc-button', n_clicks=0,
                        style={
                            'backgroundColor': '#4CAF50', 'color': 'white',
                            'border': 'none', 'padding': '10px 30px',
                            'fontSize': '16px', 'cursor': 'pointer',
                            'borderRadius': '4px',
                        }),
        ], style={'textAlign': 'center', 'marginBottom': '10px'}),
    ], style={'padding': '15px', 'backgroundColor': '#f9f9f9',
              'borderRadius': '8px', 'marginBottom': '20px'}),

    # --- Метрики-карточки ---
    html.Div(id='metrics-cards', style={'display': 'flex', 'gap': '15px', 'marginBottom': '20px'}),

    # --- Графики ---
    dcc.Graph(id='event-study-graph', style={'height': '600px'}),

    # Сообщение об ошибке
    html.Div(id='error-msg', style={'color': 'red', 'textAlign': 'center', 'marginTop': '10px'}),
])


# === Callback 1: обновить список событий при смене тикера ===
@app.callback(
    Output('event-dropdown', 'options'),
    Output('event-dropdown', 'value'),
    Input('ticker-dropdown', 'value'),
)
def update_events_dropdown(ticker):
    evs = events_by_ticker.get(ticker, [])
    options = [{'label': _event_label(ev), 'value': _event_value(ev)} for ev in evs]
    value = _event_value(evs[0]) if evs else None
    return options, value


# === Callback 2: расчёт и отрисовка ===
@app.callback(
    Output('event-study-graph', 'figure'),
    Output('metrics-cards', 'children'),
    Output('error-msg', 'children'),
    Input('calc-button', 'n_clicks'),
    State('ticker-dropdown', 'value'),
    State('event-dropdown', 'value'),
    State('model-dropdown', 'value'),
    State('ew-before', 'value'),
    State('ew-after', 'value'),
    State('est-length', 'value'),
    prevent_initial_call=True,
)
def calculate(n_clicks, ticker, event_val, model_type, ew_before, ew_after, est_length):
    if not event_val:
        return no_update, no_update, 'Выберите событие'

    # Парсим event_val: "TICKER_YYYY-MM-DD"
    parts = event_val.split('_', 1)
    event_date_str = parts[1]
    target_date = date.fromisoformat(event_date_str)

    # Находим объект Event
    evs = events_by_ticker.get(ticker, [])
    event = None
    for ev in evs:
        if ev.event_date == target_date:
            event = ev
            break

    if event is None:
        return no_update, no_update, f'Событие не найдено: {ticker} {target_date}'

    # Конфигурация
    config = EventStudyConfig(
        model=model_type,
        event_window=(-ew_before, ew_after),
        estimation_window=est_length,
    )

    # Расчёт
    result = runner.analyze_single_event(event, config)

    if result is None:
        return no_update, no_update, 'Недостаточно данных для расчёта. Попробуйте уменьшить окна.'

    # --- Данные для графика ---
    n_days = result.n_days
    ar = np.array(result.ar)
    car = np.cumsum(ar) * 100  # в проценты
    days = list(range(-ew_before, -ew_before + n_days))

    # Объёмы торгов
    vol_series = volumes.get(ticker)
    vol_days = []
    vol_values = []
    if vol_series is not None:
        t0 = pd.Timestamp(target_date)
        trading_days = vol_series.index.sort_values()
        idx0 = trading_days.searchsorted(t0, side='left')
        for d in days:
            pos = idx0 + d
            if 0 <= pos < len(trading_days):
                dt = trading_days[pos]
                if dt in vol_series.index:
                    vol_days.append(d)
                    vol_values.append(vol_series[dt])

    # --- График ---
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=['Cumulative Abnormal Return (CAR)', 'Объём торгов'],
    )

    # CAR линия
    fig.add_trace(go.Scatter(
        x=days, y=car,
        mode='lines+markers',
        line=dict(color='steelblue', width=2.5),
        marker=dict(size=4),
        name='CAR, %',
        hovertemplate='t=%{x}: CAR=%{y:.3f}%<extra></extra>',
    ), row=1, col=1)

    # Нулевая линия
    fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1, row=1, col=1)

    # Вертикальная линия t=0
    fig.add_vline(x=0, line_dash='dash', line_color='crimson', line_width=1.5,
                  annotation_text='t=0', annotation_position='top right')

    # Объём barchart
    if vol_days:
        colors = ['rgba(100,149,237,0.5)' if d <= 0 else 'rgba(220,80,80,0.5)' for d in vol_days]
        fig.add_trace(go.Bar(
            x=vol_days, y=vol_values,
            marker_color=colors,
            name='Объём',
            hovertemplate='t=%{x}: %{y:,.0f}<extra></extra>',
            showlegend=False,
        ), row=2, col=1)

    fig.update_layout(
        template='plotly_white',
        height=550,
        margin=dict(l=50, r=30, t=60, b=40),
        xaxis2_title='Торговые дни от события',
        yaxis_title='CAR, %',
        yaxis2_title='Объём',
        showlegend=False,
        hovermode='x unified',
    )

    # --- Метрики-карточки ---
    car_total = result.car * 100  # в проценты

    # avg_return: средняя цена после vs до (простая аппроксимация через AR)
    ar_arr = np.array(result.ar)
    zero_idx = ew_before  # позиция t=0 в массиве days
    ar_before = ar_arr[:zero_idx] if zero_idx > 0 else np.array([])
    ar_after = ar_arr[zero_idx:] if zero_idx < len(ar_arr) else np.array([])

    # vol_ratio и volume_ratio
    if vol_series is not None and vol_days:
        vols_before = [v for d, v in zip(vol_days, vol_values) if d < 0]
        vols_after = [v for d, v in zip(vol_days, vol_values) if d > 0]
        mean_vol_before = np.mean(vols_before) if vols_before else 0
        mean_vol_after = np.mean(vols_after) if vols_after else 0
        volume_ratio = round(mean_vol_after / mean_vol_before, 2) if mean_vol_before > 0 else float('nan')
    else:
        volume_ratio = float('nan')

    # Волатильность: std AR до и после
    vol_ret_before = float(np.std(ar_before) * 100) if len(ar_before) > 1 else 0
    vol_ret_after = float(np.std(ar_after) * 100) if len(ar_after) > 1 else 0
    vol_ratio_val = round(vol_ret_after / vol_ret_before, 2) if vol_ret_before > 0 else float('nan')

    def _card(title, value, color='#333'):
        return html.Div([
            html.Div(title, style={'fontSize': '13px', 'color': '#888', 'marginBottom': '4px'}),
            html.Div(value, style={'fontSize': '22px', 'fontWeight': 'bold', 'color': color}),
        ], style={
            'flex': '1', 'textAlign': 'center', 'padding': '12px 8px',
            'backgroundColor': 'white', 'borderRadius': '8px',
            'boxShadow': '0 1px 3px rgba(0,0,0,0.12)',
        })

    car_color = '#2e7d32' if car_total >= 0 else '#c62828'
    cards = [
        _card('CAR, %', f'{car_total:+.3f}%', car_color),
        _card('Коэф. волатильности', f'{vol_ratio_val}'),
        _card('Коэф. объёма', f'{volume_ratio}'),
        _card('Дней в окне', f'{n_days}'),
    ]

    return fig, cards, ''


port = _find_free_port()
app.run(port=port, jupyter_mode='inline')